- **Name:** 18_dataframe_optimizations
- **Author:** Shamas Imran
- **Desciption:** Optimizing DataFrame performance
- **Date:** 19-Aug-2025
<!--
REVISION HISTORY
Version          Date        Author           Desciption
01           19-Aug-2025   Shamas Imran       Demonstrated caching and persistence  
                                              Used repartition and coalesce  
                                              Analyzed execution plan with explain  
-->

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("DataFrameOptimizations").getOrCreate()

# Sample DataFrames
df_large = spark.range(0, 1_000_000).withColumn("value", (F.rand() * 100).cast("int"))
df_small = spark.createDataFrame([(i, f"cat_{i}") for i in range(100)], ["id", "category"])

spark.createDataFrame(df_large.tail(20)).show(truncate=False)
df_small.show()

StatementMeta(, 5d6ad8b2-194c-4f54-98ff-0160199ad166, 4, Finished, Available, Finished)

+------+-----+
|id    |value|
+------+-----+
|999980|3    |
|999981|95   |
|999982|67   |
|999983|3    |
|999984|32   |
|999985|78   |
|999986|95   |
|999987|85   |
|999988|1    |
|999989|5    |
|999990|23   |
|999991|94   |
|999992|64   |
|999993|29   |
|999994|10   |
|999995|78   |
|999996|18   |
|999997|32   |
|999998|84   |
|999999|9    |
+------+-----+

+---+--------+
| id|category|
+---+--------+
|  0|   cat_0|
|  1|   cat_1|
|  2|   cat_2|
|  3|   cat_3|
|  4|   cat_4|
|  5|   cat_5|
|  6|   cat_6|
|  7|   cat_7|
|  8|   cat_8|
|  9|   cat_9|
| 10|  cat_10|
| 11|  cat_11|
| 12|  cat_12|
| 13|  cat_13|
| 14|  cat_14|
| 15|  cat_15|
| 16|  cat_16|
| 17|  cat_17|
| 18|  cat_18|
| 19|  cat_19|
+---+--------+
only showing top 20 rows



In [3]:
                                        # ---------------------------------------------------------
                                        # 1. Repartition vs Coalesce
                                        # ---------------------------------------------------------
# Repartition increases or decreases partitions (full shuffle)

print("Number of partitions Before repartition:", df_large.rdd.getNumPartitions())
df_repart = df_large.repartition(10)
print("Number of partitions after repartition:", df_repart.rdd.getNumPartitions())

# Coalesce only decreases partitions (avoids full shuffle if possible)
df_coalesce = df_large.coalesce(2)
print("Number of partitions after coalesce:", df_coalesce.rdd.getNumPartitions())

print("df_small partitions:", df_small.rdd.getNumPartitions())

StatementMeta(, 5d6ad8b2-194c-4f54-98ff-0160199ad166, 5, Finished, Available, Finished)

Number of partitions Before repartition: 8
Number of partitions after repartition: 10
Number of partitions after coalesce: 2
df_small partitions: 8


In [ ]:
                                        # ---------------------------------------------------------
                                        # 2. Caching and Persistence
                                        # ---------------------------------------------------------
# Cache: stores the DataFrame in memory (default MEMORY_AND_DISK)
df_cached = df_large.cache()
print("First count triggers caching:", df_cached.count())  # first action
print("Second count reads from cache:", df_cached.count())

# Persist: allows specifying storage levels
from pyspark import StorageLevel
df_persist = df_large.persist(StorageLevel.MEMORY_ONLY)

# StorageLevel.MEMORY_ONLY ==> Store as deserialized objects in memory. Recompute partitions if not enough RAM. (Fastest but uses more memory)
# StorageLevel.MEMORY_AND_DISK ==> Store in memory, spill to disk if needed. (More reliable for large data)
# StorageLevel.MEMORY_ONLY_SER ==> Store as serialized objects in memory. (Saves memory, slightly slower)
# StorageLevel.MEMORY_AND_DISK_SER ==> Serialize and store in memory or disk if needed. (Balanced)
# StorageLevel.DISK_ONLY ==> Store only on disk. (Slowest but least memory usage)

In [ ]:
# ---------------------------------------------------------
# 3. Broadcast Join Optimization
# ---------------------------------------------------------
# Broadcast small DataFrame to avoid shuffle join
df_join = df_large.join(F.broadcast(df_small), df_large.value == df_small.id, "inner")
df_join.explain()  # Check execution plan for BroadcastHashJoin

In [6]:
# ---------------------------------------------------------
# 4. Handling Data Skew
# ---------------------------------------------------------
# Simulate skew: most rows have value = 1
df_skewed = df_large.withColumn("key", F.when(F.rand() < 0.9, 1).otherwise(F.col("value")))
df_skewed.show()

# Skew mitigation: add a "salt" key to spread data
df_skewed_salted = df_skewed.withColumn("salt", (F.rand() * 10).cast("int"))
df_skewed_salted.show()

df_small_salted = df_small.withColumn("salt", F.expr("explode(sequence(0,9))"))
df_small_salted.show()

df_salted_join = df_skewed_salted.join(
    df_small_salted,
    (df_skewed_salted.key == df_small_salted.id) & (df_skewed_salted.salt == df_small_salted.salt),
    "inner"
)

StatementMeta(, 5d6ad8b2-194c-4f54-98ff-0160199ad166, 8, Finished, Available, Finished)

+---+-----+---+
| id|value|key|
+---+-----+---+
|  0|   51|  1|
|  1|   39|  1|
|  2|   21| 21|
|  3|   12|  1|
|  4|   52|  1|
|  5|   88|  1|
|  6|   18|  1|
|  7|   62|  1|
|  8|   60|  1|
|  9|   39|  1|
| 10|   22|  1|
| 11|   63|  1|
| 12|   39|  1|
| 13|   72|  1|
| 14|   82|  1|
| 15|   31|  1|
| 16|   81| 81|
| 17|   78| 78|
| 18|   98|  1|
| 19|    7|  1|
+---+-----+---+
only showing top 20 rows

+---+-----+---+----+
| id|value|key|salt|
+---+-----+---+----+
|  0|   51|  1|   9|
|  1|   39|  1|   9|
|  2|   21| 21|   6|
|  3|   12|  1|   9|
|  4|   52|  1|   1|
|  5|   88|  1|   0|
|  6|   18|  1|   3|
|  7|   62|  1|   2|
|  8|   60|  1|   0|
|  9|   39|  1|   5|
| 10|   22|  1|   4|
| 11|   63|  1|   1|
| 12|   39|  1|   2|
| 13|   72|  1|   7|
| 14|   82|  1|   8|
| 15|   31|  1|   1|
| 16|   81| 81|   9|
| 17|   78| 78|   4|
| 18|   98|  1|   8|
| 19|    7|  1|   5|
+---+-----+---+----+
only showing top 20 rows

+---+--------+----+
| id|category|salt|
+---+--------+----+


In [8]:
# ---------------------------------------------------------
# 5. Measuring Execution Plans
# ---------------------------------------------------------
# Explain gives logical and physical execution plans
df_salted_join.explain(True)  # Detailed execution plan


StatementMeta(, 5d6ad8b2-194c-4f54-98ff-0160199ad166, 10, Finished, Available, Finished)

== Parsed Logical Plan ==
Join Inner, ((cast(key#856 as bigint) = id#731L) AND (salt#873 = salt#896))
:- Project [id#726L, value#728, key#856, cast((rand(-9177878709431959483) * cast(10 as double)) as int) AS salt#873]
:  +- Project [id#726L, value#728, CASE WHEN (rand(167888447934510300) < 0.9) THEN 1 ELSE value#728 END AS key#856]
:     +- Project [id#726L, cast((rand(-4159216817967408431) * cast(100 as double)) as int) AS value#728]
:        +- Range (0, 1000000, step=1, splits=Some(8))
+- Project [id#731L, category#732, salt#896]
   +- Generate explode(sequence(0, 9, None, Some(UTC))), false, [salt#896]
      +- LogicalRDD [id#731L, category#732], false

== Analyzed Logical Plan ==
id: bigint, value: int, key: int, salt: int, id: bigint, category: string, salt: int
Join Inner, ((cast(key#856 as bigint) = id#731L) AND (salt#873 = salt#896))
:- Project [id#726L, value#728, key#856, cast((rand(-9177878709431959483) * cast(10 as double)) as int) AS salt#873]
:  +- Project [id#726L, val